In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

CDC_PATH = "/Volumes/claims/raw/landing/raw/policy_changes_cdc/*.json"

# 1. Load all CDC events; parse to proper types
policy_events = (
    spark.read.json(CDC_PATH)
    .withColumn("effective_from", F.to_date("effective_from"))
    .withColumn("op_ts", F.to_timestamp("op_ts"))
)

# 2. SCD2 dating: order by business time per member+policy
policy_window = Window.partitionBy("member_id", "policy_id").orderBy("effective_from")

policy_scd2 = (
    policy_events
    # next event's effective_from closes this row
    .withColumn("next_effective_from", F.lead("effective_from").over(policy_window))
    # DELETE only terminates the policy history — it is not a version of its own
    .filter(F.col("op") != "DELETE")
    # surrogate key: deterministic hash of the natural key + version date
    .withColumn("policy_sk", F.md5(F.concat_ws("||", "member_id", "policy_id", "effective_from")))
    .withColumn("valid_from", F.col("effective_from"))
    .withColumn("valid_to", F.date_sub(F.col("next_effective_from"), 1))
    # current only when nothing supersedes this row (open-ended)
    .withColumn("is_current", F.col("next_effective_from").isNull())
    .select("policy_sk", "member_id", "policy_id", "plan_code", "valid_from", "valid_to", "is_current")
)

# 3. Publish to claims.silver
policy_scd2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("claims.silver.dim_policy_scd2")

print(f"dim_policy_scd2 rows written: {policy_scd2.count()}")
print(f"  current (open) rows : {policy_scd2.filter('is_current').count()}")
print(f"  closed history rows : {policy_scd2.filter('NOT is_current').count()}")

# 4. Sanity checks against SCD2 invariants
verified = spark.table("claims.silver.dim_policy_scd2")

assert verified.count() == 661 - 10, "row count mismatch: 661 events minus 10 DELETE tombstones"
assert verified.select("policy_sk").distinct().count() == verified.count(), "policy_sk not unique"
assert verified.filter("is_current AND valid_to IS NOT NULL").count() == 0, "current row has valid_to"
assert verified.filter("NOT is_current AND valid_to IS NULL").count() == 0, "closed row has no valid_to"
assert verified.filter("is_current").groupBy("member_id").count().filter("count > 1").count() == 0, "member with multiple current rows"

# continuity: each closed row must end the day before the next row starts (no gaps, no overlaps)
verify_window = Window.partitionBy("member_id", "policy_id").orderBy("valid_from")
gap_check = verified.withColumn("prev_to", F.lag("valid_to").over(verify_window))
assert gap_check.filter(F.col("prev_to").isNotNull() & (F.col("prev_to") != F.date_sub("valid_from", 1))).count() == 0, "gap/overlap between rows"

print("All SCD2 invariants verified.")